# CSE 151B — SFT + LoRA Fine-Tuning Notebook

This notebook fine-tunes **Qwen3-4B** (or Qwen3-4B-Instruct) with **LoRA** on the public competition dataset,
then runs inference the same way as the starter notebook.

Pipeline:
1. Install dependencies (unsloth / PEFT + TRL)
2. Load & format the public dataset as SFT examples
3. Load the base model with 4-bit QLoRA
4. Train with `SFTTrainer`
5. Save the merged model (or just the LoRA adapter)
6. Run inference & score (same as starter notebook)

> **Tip**: Use `unsloth` for 2x faster training + lower VRAM. If unsloth doesn't support
> your CUDA version, fall back to plain `peft` + `trl` (Section 1B).

## 1A. Install with Unsloth (recommended — faster, less VRAM)

In [ ]:
# Run once, then comment out
import sys
!{sys.executable} -m pip install -q \
    'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git' \
    unsloth_zoo \
    trl peft accelerate bitsandbytes \
    sympy numpy tqdm requests 'antlr4-python3-runtime==4.11.1'
print('Done. Restart kernel now if this was the first install.')

## 1B. Install without Unsloth (fallback)

In [ ]:
# USE THIS if unsloth fails to install
# import sys
# !{sys.executable} -m pip install -q \
#     transformers>=4.40 peft>=0.11 trl>=0.8 accelerate bitsandbytes \
#     sympy numpy tqdm requests 'antlr4-python3-runtime==4.11.1'
# print('Done.')

## 2. Imports & Configuration

In [ ]:
import json, re, sys, os
from pathlib import Path
from typing import Optional

# ── Config ────────────────────────────────────────────────────────────────────
BASE_MODEL      = "Qwen/Qwen3-4B"          # HF model ID — change to local path if already downloaded
DATA_PATH       = "data/public.jsonl"       # same dataset as starter notebook
OUTPUT_DIR      = "lora_output/qwen3-4b-math"  # where adapter weights are saved
MERGED_DIR      = "merged_model/qwen3-4b-math" # where the fully-merged model is saved
RESULT_PATH     = "results/lora_results.jsonl"

# LoRA hypers
LORA_R          = 16       # rank — increase to 32/64 for more capacity, costs more VRAM
LORA_ALPHA      = 32       # typically 2×rank
LORA_DROPOUT    = 0.05
TARGET_MODULES  = [        # Qwen3 attention + MLP projections
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

# Training hypers
MAX_SEQ_LEN     = 2048     # truncate long examples; bump to 4096 if VRAM allows
BATCH_SIZE      = 2        # per-device batch size
GRAD_ACCUM      = 8        # effective batch = BATCH_SIZE × GRAD_ACCUM = 16
NUM_EPOCHS      = 3        # 2–4 is usually enough on this dataset size
LR              = 2e-4
WARMUP_RATIO    = 0.05
USE_UNSLOTH     = True     # set False if you installed the fallback above

# Inference (after fine-tuning)
LM_STUDIO_URL   = "http://localhost:1234"
LM_STUDIO_MODEL = "qwen3-4b-math"         # update to whatever name you give the merged model in LM Studio
MAX_TOKENS      = 16384

print(f"Base model : {BASE_MODEL}")
print(f"Output dir : {OUTPUT_DIR}")
print(f"LoRA rank  : {LORA_R}")

## 3. Load & Format Dataset

We convert each public-set example into a chat-formatted training example.
The target (what the model learns to produce) is a chain-of-thought solution
ending in `\boxed{answer}`.

**Train/val split**: 90 % train, 10 % validation.

In [ ]:
import random
from datasets import Dataset

# ── System prompts (same as starter) ─────────────────────────────────────────
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. Solve the problem step-by-step.\n\n"
    "═══ FINAL ANSWER FORMAT — THE MOST IMPORTANT RULE ═══\n"
    "At the very LAST LINE of your response, place ALL answers inside exactly ONE \\boxed{}.\n"
    "  DO:     \\boxed{380, 315, 13, 310}  (all parts, comma-separated, one box)\n"
    "  DO:     \\boxed{5/8}  (single answer)\n"
    "  DON'T:  box each sub-answer in a separate \\boxed{} throughout the solution\n"
    "Even if you use \\boxed{} for intermediate steps during working, you MUST finish with "
    "a single combined \\boxed{a, b, c} on the very last line — all answers, in the order asked.\n"
    "Never leave \\boxed{} empty.\n\n"
    "═══ EXACT FORM RULES ═══\n"
    "1. SYMBOLIC OVER NUMERIC: write \\arctan(4.76) not 1.3635\n"
    "2. DECIMAL PRECISION: at minimum 6 significant digits\n"
    "3. FRACTIONS: use exact fractions (5/8) for rational results.\n"
    "4. ORDER: answer multi-part questions in the exact order asked.\n"
    "Before writing the final \\boxed{}, verify your answer. Commit to your best answer."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. "
    "Read the problem and the answer choices carefully, then select the single best answer.\n\n"
    "STEP 1 Solve: Work through the problem step-by-step.\n"
    "STEP 2 Match: Compare your result against every option.\n"
    "STEP 3 Commit: Trust your derivation.\n\n"
    "You MUST always pick one of the given letters. "
    "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
)


def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, question


def build_target(item: dict) -> str:
    """Construct the assistant turn we want the model to learn.
    
    For SFT we only have the gold answer, not a full chain-of-thought.
    We therefore teach the model the correct *final line* format.
    If you have richer solution traces (e.g. from GPT-4o), replace this.
    """
    answer = item["answer"]
    if item.get("options"):  # MCQ
        return f"The correct answer is \\boxed{{{answer.strip().upper()}}}"
    # Free-form — may be a list
    if isinstance(answer, list):
        joined = ", ".join(str(a) for a in answer)
        return f"Therefore the answer is \\boxed{{{joined}}}"
    return f"Therefore the answer is \\boxed{{{answer}}}"


def item_to_chat(item: dict) -> dict:
    """Convert one dataset record to a chat-format dict with a 'text' field."""
    system, user = build_prompt(item["question"], item.get("options"))
    assistant     = build_target(item)
    # Qwen3 chat template expects: system / user / assistant
    return {
        "messages": [
            {"role": "system",    "content": system},
            {"role": "user",      "content": user},
            {"role": "assistant", "content": assistant},
        ]
    }


# ── Load data ─────────────────────────────────────────────────────────────────
raw_data = [json.loads(line) for line in open(DATA_PATH)]
random.seed(42)
random.shuffle(raw_data)

split = int(0.9 * len(raw_data))
train_items = raw_data[:split]
val_items   = raw_data[split:]

train_dataset = Dataset.from_list([item_to_chat(x) for x in train_items])
val_dataset   = Dataset.from_list([item_to_chat(x) for x in val_items])

print(f"Train: {len(train_dataset)}  |  Val: {len(val_dataset)}")
print("\nSample training record:")
print(json.dumps(train_dataset[0]["messages"], indent=2)[:600], "...")

## 4. Load Model + LoRA

### 4A — with Unsloth (recommended)

In [ ]:
if USE_UNSLOTH:
    from unsloth import FastLanguageModel
    import torch

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name   = BASE_MODEL,
        max_seq_length = MAX_SEQ_LEN,
        dtype        = None,          # auto-detect: bf16 on Ampere+, fp16 otherwise
        load_in_4bit = True,          # QLoRA — halves VRAM vs fp16
    )

    model = FastLanguageModel.get_peft_model(
        model,
        r                = LORA_R,
        lora_alpha       = LORA_ALPHA,
        lora_dropout     = LORA_DROPOUT,
        target_modules   = TARGET_MODULES,
        bias             = "none",
        use_gradient_checkpointing = "unsloth",  # saves ~30% VRAM
        random_state     = 42,
        use_rslora       = False,
    )

    print(model.print_trainable_parameters())
    print("Unsloth model ready.")

### 4B — without Unsloth (fallback, plain PEFT)

In [ ]:
if not USE_UNSLOTH:
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    from peft import LoraConfig, get_peft_model, TaskType

    bnb_config = BitsAndBytesConfig(
        load_in_4bit               = True,
        bnb_4bit_use_double_quant  = True,
        bnb_4bit_quant_type        = "nf4",
        bnb_4bit_compute_dtype     = torch.bfloat16,
    )

    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config = bnb_config,
        device_map          = "auto",
        trust_remote_code   = True,
    )

    lora_config = LoraConfig(
        r              = LORA_R,
        lora_alpha     = LORA_ALPHA,
        lora_dropout   = LORA_DROPOUT,
        target_modules = TARGET_MODULES,
        bias           = "none",
        task_type      = TaskType.CAUSAL_LM,
    )

    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    print("PEFT model ready.")

## 5. Apply Chat Template

Convert messages → raw text using the model's built-in chat template.

In [ ]:
def apply_chat_template(examples):
    texts = []
    for msgs in examples["messages"]:
        text = tokenizer.apply_chat_template(
            msgs,
            tokenize         = False,
            add_generation_prompt = False,  # includes EOS — we want that for SFT
        )
        texts.append(text)
    return {"text": texts}


train_dataset = train_dataset.map(apply_chat_template, batched=True,
                                   remove_columns=["messages"])
val_dataset   = val_dataset.map(apply_chat_template,   batched=True,
                                 remove_columns=["messages"])

print("Sample text (first 500 chars):")
print(train_dataset[0]["text"][:500])

## 6. Train with SFTTrainer

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = NUM_EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    warmup_ratio                = WARMUP_RATIO,
    learning_rate               = LR,
    lr_scheduler_type           = "cosine",
    fp16                        = not torch.cuda.is_bf16_supported(),
    bf16                        = torch.cuda.is_bf16_supported(),
    optim                       = "adamw_8bit",  # 8-bit Adam reduces VRAM
    logging_steps               = 10,
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    save_total_limit            = 2,
    load_best_model_at_end      = True,
    metric_for_best_model       = "eval_loss",
    report_to                   = "none",  # set to "wandb" if you have it configured
    seed                        = 42,
)

trainer = SFTTrainer(
    model           = model,
    tokenizer       = tokenizer,
    train_dataset   = train_dataset,
    eval_dataset    = val_dataset,
    dataset_text_field = "text",
    max_seq_length  = MAX_SEQ_LEN,
    dataset_num_proc = 2,
    args            = training_args,
)

print(f"Starting training for {NUM_EPOCHS} epoch(s) …")
trainer_stats = trainer.train()
print(f"Training complete. Loss: {trainer_stats.training_loss:.4f}")

## 7. Save Adapter & Merge

- **Adapter only** (small, fast to save): load base model + adapter at inference time
- **Merged model** (self-contained): load with LM Studio like any other model

For the competition you likely want the merged model so you can serve it via LM Studio.

In [ ]:
# ── Save LoRA adapter (always do this first) ──────────────────────────────────
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Adapter saved → {OUTPUT_DIR}")

In [ ]:
# ── Merge adapter into base model weights (for LM Studio / vLLM inference) ───
Path(MERGED_DIR).mkdir(parents=True, exist_ok=True)

if USE_UNSLOTH:
    # Unsloth has a convenience method that handles dequantisation
    model.save_pretrained_merged(
        MERGED_DIR, tokenizer,
        save_method = "merged_16bit",  # or "merged_4bit" to keep quantised
    )
else:
    # Standard PEFT merge
    merged = model.merge_and_unload()
    merged.save_pretrained(MERGED_DIR)
    tokenizer.save_pretrained(MERGED_DIR)

print(f"Merged model saved → {MERGED_DIR}")
print("\nNext: load this folder in LM Studio (drag-and-drop or point to the path).")

## 8. (Optional) Direct Inference Without LM Studio

If you want to run inference directly in this notebook (no LM Studio required),
use the cell below. Otherwise skip to Section 9 for LM Studio inference.

In [ ]:
# Direct in-notebook inference (no LM Studio)
# Uncomment if you want to skip LM Studio.

# if USE_UNSLOTH:
#     FastLanguageModel.for_inference(model)  # enable faster inference mode

# def generate_direct(question: str, options=None, max_new_tokens=2048) -> str:
#     system, user = build_prompt(question, options)
#     messages = [
#         {"role": "system",    "content": system},
#         {"role": "user",      "content": user},
#     ]
#     input_ids = tokenizer.apply_chat_template(
#         messages, return_tensors="pt", add_generation_prompt=True
#     ).to(model.device)
#     with torch.no_grad():
#         output = model.generate(
#             input_ids,
#             max_new_tokens   = max_new_tokens,
#             temperature      = 0.2,
#             top_p            = 0.95,
#             do_sample        = True,
#             pad_token_id     = tokenizer.eos_token_id,
#         )
#     new_tokens = output[0][input_ids.shape[-1]:]
#     return tokenizer.decode(new_tokens, skip_special_tokens=True)

print("Direct inference cell is commented out. Uncomment to use.")

## 9. Inference via LM Studio (same as starter)

Load the merged model in LM Studio, start the server, then run the cells below.
This mirrors the starter notebook's inference loop exactly.

In [ ]:
import json, re, sys, requests
from pathlib import Path
from tqdm import tqdm

data = [json.loads(line) for line in open(DATA_PATH)]
print(f"Loaded {len(data)} questions")

# Verify LM Studio is up
try:
    r = requests.get(f"{LM_STUDIO_URL}/v1/models", timeout=5)
    r.raise_for_status()
    models = [m["id"] for m in r.json().get("data", [])]
    print(f"LM Studio running. Models: {models}")
    if LM_STUDIO_MODEL not in models:
        print(f"WARNING: '{LM_STUDIO_MODEL}' not found. Loaded: {models}")
except requests.exceptions.ConnectionError:
    print(f"ERROR: Cannot reach LM Studio at {LM_STUDIO_URL}")

In [ ]:
sys.path.insert(0, '.')
from judger import Judger
judger = Judger(strict_extract=False)

def extract_letter(text: str) -> str:
    m = re.search(r'\\boxed\{([A-Za-z])\}', text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r'\b([A-Z])\b', text.upper())
    return matches[-1] if matches else ''

def score_response(item: dict, response: str) -> bool:
    if item.get('options'):
        return extract_letter(response) == item['answer'].strip().upper()
    gold_list = item['answer'] if isinstance(item['answer'], list) else [item['answer']]
    return judger.auto_judge(response, gold_list, options=[[]]*len(gold_list))

def generate_response(question: str, options=None) -> str:
    system, user = build_prompt(question, options)
    r = requests.post(
        f"{LM_STUDIO_URL}/v1/chat/completions",
        json={
            "model":       LM_STUDIO_MODEL,
            "messages":    [
                {"role": "system", "content": system},
                {"role": "user",   "content": user},
            ],
            "temperature": 0.2,
            "top_p":       0.95,
            "max_tokens":  MAX_TOKENS,
            "thinking": {"type": "enabled", "budget_tokens": 15000},
        },
        timeout=600,
    )
    if not r.ok:
        raise RuntimeError(f"LM Studio error {r.status_code}: {r.text}")
    return r.json()["choices"][0]["message"]["content"]


out_path = Path(RESULT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

done_ids = set()
results  = []
if out_path.exists():
    with open(out_path) as f:
        for line in f:
            rec = json.loads(line)
            done_ids.add(rec['id'])
            results.append(rec)
    print(f"Resuming: {len(done_ids)} done, {len(data)-len(done_ids)} remaining")
else:
    print("Starting fresh run")

with open(out_path, 'a') as f:
    for item in tqdm(data, desc="Generating"):
        if item['id'] in done_ids:
            continue
        try:
            response = generate_response(item["question"], item.get("options"))
            correct  = score_response(item, response)
        except Exception as e:
            print(f"\nERROR on id={item['id']}: {e}")
            response, correct = "", False
        record = {'id': item['id'], 'is_mcq': bool(item.get('options')),
                  'gold': item['answer'], 'response': response, 'correct': correct}
        results.append(record)
        f.write(json.dumps(record) + '\n')
        f.flush()

## 10. Score Summary

In [ ]:
results = [json.loads(line) for line in open(RESULT_PATH)]
mcq_res  = [r for r in results if  r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

print("=" * 50)
print("LORA FINE-TUNED RESULTS")
print("=" * 50)
print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
print("=" * 50)

---
## Tips & Next Steps

### Getting better SFT data (most impactful)
The biggest weakness of this notebook is that `build_target()` only teaches the model the
**final answer format**, not a full chain-of-thought.  To fix this:

1. **Generate traces with the base model** — run Qwen3-4B on the public set, keep
   only the responses where it answered correctly, and use those as training targets.
   This is "rejection sampling" and tends to help a lot.
2. **Use GPT-4o / Claude to write solutions** for a subset of problems, then SFT on those.

### Hyperparameter knobs
| Parameter | Effect | Try |
|---|---|---|
| `LORA_R` | Capacity vs VRAM | 8 → 16 → 32 |
| `NUM_EPOCHS` | Overfitting risk | 2 → 3 → 4 |
| `LR` | Stability | 1e-4 → 2e-4 → 5e-4 |
| `MAX_SEQ_LEN` | Long-problem coverage | 2048 → 4096 |

### Combining SFT + RL
After SFT, consider GRPO/PPO with the `judger` as a reward signal — this is the
standard recipe for math reasoning (see DeepSeek-R1, STILL-2).

### VRAM estimate (A100 40 GB)
- QLoRA 4-bit + r=16: ~14 GB
- QLoRA 4-bit + r=64: ~18 GB
- Full fp16 fine-tune (no LoRA): ~32 GB